# 03 - Gold checks: label store and feature store

Gold is what training, inference and monitoring read (contracts §3–4): one row per community area × hour, the full grid with zeros kept. This notebook checks:

1. the label store: the full grid, crash totals against silver, and the snapshots;
2. the feature store: one row per area × target hour, `feature_cutoff = hour_start`, missing values only where expected;
3. point in time: a sample of feature values recounted straight from silver, using only records that arrived before each hour.

In [1]:
import os

import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import yaml
from pyspark.sql import Window

# run from the repo root, where main.py, config/ and data/ are
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import main
from utils.gold import hour_grid

with open("config/pipeline.yaml") as f:
    config = yaml.safe_load(f)
spark = main.get_spark(config)

labels_all = spark.read.parquet("datamart/gold/label_store")
features = spark.read.parquet("datamart/gold/feature_store")
silver_crashes = spark.read.parquet("datamart/silver/crashes")

# readers take each hour's latest label snapshot (contracts §3)
latest = Window.partitionBy("community_area", "hour_start").orderBy(F.col("label_snapshot_date").desc())
labels = labels_all.withColumn("n", F.row_number().over(latest)).filter("n = 1").drop("n")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 08:20:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1 - Label store

The bulk history labels every hour from 2018-03-01 to the end of 2022. Each monthly run labels its own month, then labels the month before again, with one more month of late reports. So every hour has two snapshots, except December 2022: only its own run labels it, because the next run is live.

In [2]:
hours = hour_grid("2018-03-01 00:00", "2022-12-31 23:00")
print(f"Chicago clock hours: {len(hours):,} -> expected rows: 77 x {len(hours):,} = {77 * len(hours):,}")
print(f"label rows (latest snapshot): {labels.count():,}")
print("duplicate keys:", labels.groupBy("community_area", "hour_start").count().filter("count > 1").count())

totals = labels.agg(F.sum("crash_count").alias("crashes"), F.sum("crash_count_injury_tow").alias("injury_tow"),
                    F.mean("crash_count").alias("mean_per_area_hour"),
                    F.mean((F.col("crash_count") == 0).cast("int")).alias("share_zero")).first()
print(f"crashes labelled: {totals.crashes:,} (injury/tow {totals.injury_tow:,}); "
      f"mean per area-hour {totals.mean_per_area_hour:.3f}; area-hours with no crash {totals.share_zero:.1%}")

Chicago clock hours: 42,403 -> expected rows: 77 x 42,403 = 3,265,031
label rows (latest snapshot): 3,265,031
duplicate keys: 0
crashes labelled: 523,173 (injury/tow 151,374); mean per area-hour 0.160; area-hours with no crash 86.5%


In [3]:
# silver holds every located crash reported before go-live. The labels miss only those reported
# more than a month after the end of the month they happened in (after the month's last rebuild)
occurred = F.date_trunc("month", "crash_date")
rebuild_cutoff = F.add_months(occurred, 2)  # the run for the next month sees reports until its end
late = silver_crashes.filter(F.col("_arrived_at") >= rebuild_cutoff)
print(f"silver crashes: {silver_crashes.count():,}")
print(f"reported after their month's last rebuild: {late.count():,}")
print(f"silver minus those: {silver_crashes.count() - late.count():,}")

silver crashes: 523,611
reported after their month's last rebuild: 438
silver minus those: 523,173


In [4]:
# snapshots per month: each hour is labelled by its own month's run, then again by the next month's
(labels_all.groupBy(F.date_format("hour_start", "yyyy-MM").alias("month"))
           .agg(F.countDistinct("label_snapshot_date").alias("snapshots"), F.sum("crash_count").alias("crashes_all_snapshots"))
           .orderBy("month").toPandas().set_index("month").iloc[[0, 1, 2, -2, -1]])

,snapshots,crashes_all_snapshots
month,,
2018-03,2,18437
2018-04,2,19099
2018-05,2,21215
2022-11,2,17277
2022-12,1,8664


## 2 - Feature store

A run for `[start, end)` writes the target hours `(start, end]`: it has seen everything that arrived before `end`, so it can build features for the hour that starts at `end`. The hourly run for `[T−1, T)` writes T's 77 rows the same way (contracts §4). The first run also writes the window's first hour.

In [5]:
targets = hour_grid("2018-03-01 00:00", "2023-01-01 00:00")
print(f"target hours: {len(targets):,} -> expected rows: {77 * len(targets):,}")
print(f"feature rows: {features.count():,}")
print("duplicate keys:", features.groupBy("community_area", "hour_start").count().filter("count > 1").count())
print("rows where feature_cutoff != hour_start:", features.filter(F.col("feature_cutoff") != F.col("hour_start")).count())

feature_columns = [c for c in features.columns if c.startswith(("cal_", "crash_", "wx_"))]
nulls = features.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in feature_columns]).toPandas().T[0]
print("missing values (features not listed have none):")
nulls[nulls > 0]

target hours: 42,404 -> expected rows: 3,265,108
feature rows: 3,265,108
duplicate keys: 0
rows where feature_cutoff != hour_start: 0
missing values (features not listed have none):


crash_area_how_mean_4w    51667
crash_city_how_mean_4w    51667
wx_dew_point_c              616
wx_precip_1h_mm           35189
wx_temp_c                   231
wx_visibility_m              77
wx_wind_ms                  231
Name: 0, dtype: int64

In [6]:
features.select(feature_columns).toPandas().describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
cal_covid_regime,3265108.0,0.28,0.45,0.00,0.00,0.00,1.00,1.00
cal_hour,3265108.0,11.50,6.92,0.00,6.00,12.00,18.00,23.00
cal_is_holiday,3265108.0,0.03,0.16,0.00,0.00,0.00,0.00,1.00
cal_month,3265108.0,6.69,3.38,1.00,4.00,7.00,10.00,12.00
cal_weekday,3265108.0,4.00,2.00,1.00,2.00,4.00,6.00,7.00
crash_area_how_mean_4w,3213441.0,0.16,0.26,0.00,0.00,0.00,0.25,4.25
crash_area_notified_1h,3265108.0,0.16,0.44,0.00,0.00,0.00,0.00,9.00
crash_area_notified_24h,3265108.0,3.85,3.64,0.00,1.00,3.00,5.00,48.00
crash_area_notified_3h,3265108.0,0.48,0.87,0.00,0.00,0.00,1.00,17.00
crash_city_how_mean_4w,3213441.0,12.28,7.32,0.00,6.00,11.75,17.50,44.25


## 3 - Point in time

Recount a random sample of feature rows straight from silver with pandas, using only records with `_arrived_at < hour_start`. The values must match exactly. (The unit tests in `tests/test_gold.py` check the same rule on hand-made edge cases: a crash reported exactly at T, a weather reading at T.)

In [7]:
sample = (features.sample(fraction=0.0005, seed=611)
                  .select("community_area", "hour_start", "crash_area_notified_1h", "crash_area_notified_24h",
                          "crash_city_notified_3h", "crash_area_how_mean_4w", "wx_temp_c")
                  .toPandas())
crashes = silver_crashes.select("community_area", "crash_date", "_arrived_at").toPandas()
crashes["occurred_hour"] = crashes["crash_date"].dt.floor("h")
weather = spark.read.parquet("datamart/silver/weather").select("observed_at", "temp_c").toPandas().sort_values("observed_at")


def recount(row):
    T = pd.Timestamp(row.hour_start)
    known = crashes[crashes["_arrived_at"] < T]          # only what had arrived before T
    area = known[known["community_area"] == row.community_area]
    weeks = [T - pd.Timedelta(days=7 * k) for k in range(1, 5)]
    last_reading = weather[weather["observed_at"] < T].tail(1)
    return pd.Series({
        "crash_area_notified_1h": (area["_arrived_at"] >= T - pd.Timedelta(hours=1)).sum(),
        "crash_area_notified_24h": (area["_arrived_at"] >= T - pd.Timedelta(hours=24)).sum(),
        "crash_city_notified_3h": (known["_arrived_at"] >= T - pd.Timedelta(hours=3)).sum(),
        "crash_area_how_mean_4w": area["occurred_hour"].isin(weeks).sum() / 4
                                  if T >= pd.Timestamp("2018-03-29") else np.nan,
        "wx_temp_c": last_reading["temp_c"].iloc[0] if len(last_reading) else np.nan,
    })


expected = sample.apply(recount, axis=1)
checked = expected.columns
mismatches = ~np.isclose(sample[checked].astype(float), expected.astype(float), equal_nan=True)
print(f"rows checked: {len(sample):,}; values checked: {mismatches.size:,}; mismatches: {mismatches.sum()}")

/tmp/ipykernel_8820/959075375.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  weeks = [T - pd.Timedelta(days=7 * k) for k in range(1, 5)]
/tmp/ipykernel_8820/959075375.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  "crash_area_notified_1h": (area["_arrived_at"] >= T - pd.Timedelta(hours=1)).sum(),
/tmp/ipykernel_8820/959075375.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  "crash_area_notified_24h": (area["_arrived_at"] >= T - pd.Timedelta(hours=24)).sum(),
/tmp/ipyker

rows checked: 1,655; values checked: 8,275; mismatches: 0


**Findings:**
- **Label store:** 42,403 Chicago clock hours × 77 areas = **3,265,031 rows**, no duplicate keys. The hour count follows contracts §1: each spring-forward hour is missing, and each fall-back hour is one row. **523,173 crashes** are labelled (151,374 injury/tow): 0.160 per area-hour, as in the EDA, and **86.5% of area-hours have no crash**. The zeros are rows, not gaps.
- **Labels against silver:** silver holds 523,611 located crashes. The labels miss exactly the **438 (0.08%) reported more than a month after the end of their month**, after its last rebuild; the live pipeline's monthly rebuild is phase 3c's. December 2022 has one snapshot, because the next run is live.
- **Feature store:** 42,404 target hours (from the window's first hour to go-live at 2023-01-01 00:00) × 77 = **3,265,108 rows**, and `feature_cutoff = hour_start` in every row. Values are missing only where expected:
  - the 4-week means in the first 28 days (77 × 671 rows);
  - a weather value the last FM-15 report didn't carry (precipitation in 457 hours).
- **Point in time:** 1,655 random rows, 5 features each, recounted straight from silver using only what had arrived before T: **0 mismatches**.
- What this doesn't check: whether the features help a model. That is step 06.